# Welcome to PointNet& 3D CNNs Workshop!
In this workshop, we're going to learn how to voxelize a point cloud, and build a 3D Convolutional Neural Network to classify point clouds. When processing point clouds, voxel based approaches account for a good half of the existing approaches (with point based approaches being the second half) — and it's therefore important to understand how to implement them!

## Imports

In [75]:
# Usual Imports
import os
import sys
import json
import numpy as np
from tqdm import tqdm

# plotting library
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# append path to custom scripts
sys.path.append('../PointNet/lidar-od-scripts/gpuVersion/gpuVersion/')

# DL Imports
import torch
import torch.nn as nn

# custom imports
from visual_utils import plot_pc_data3d, plot_bboxes_3d

In [76]:
DATA_FOLDER1 = '../PointNet/shapenet-core-seg-less-subset1/Shapenetcore_benchmark/'
DATA_FOLDER2 = '../PointNet/shapenet-core-seg-less-subset2/Shapenetcore_benchmark/'

class_name_id_map = {'Airplane': 0, 'Chair': 1, 
                'Guitar': 2, 'Lamp': 3, 'Laptop': 4, 'Table': 5}

class_id_name_map = {v:k for k,v in class_name_id_map.items()}

PCD_SCENE=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), aspectmode='data')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Shapenet Core Dataset exploration

- Object Classification and part Segmentation Dataset from Point Cloud data
- [Shapenet core](https://shapenet.org/) is a subset of the full ShapeNet dataset 
- It contains single clean 3D models, manually verified category and alignment annotations. 
- 16 classes from 12 categories

In [77]:
# Load train splits for both subsets
subset1_data = json.load(open('../PointNet/shapenet-core-seg-less-subset1/Shapenetcore_benchmark/train_split.json', 'r'))
subset2_data = json.load(open('../PointNet/shapenet-core-seg-less-subset2/Shapenetcore_benchmark/train_split.json', 'r'))

# Extract class indices
subset1_class_count = np.array([x[0] for x in subset1_data])
subset2_class_count = np.array([x[0] for x in subset2_data])

# Compute class-wise counts
subset1_counts = np.bincount(subset1_class_count)
subset2_counts = np.bincount(subset2_class_count)

# Plot classwise count in train set for both subsets
train_dist_plots = [
    go.Bar(x=list(class_name_id_map.keys()), y=subset1_counts, name='Subset 1'),
    go.Bar(x=list(class_name_id_map.keys()), y=subset2_counts, name='Subset 2')
]

layout = dict(template="plotly_dark", title="Shapenet Core Train Distribution (Subsets)", title_x=0.5, barmode='group')
fig = go.Figure(data=train_dist_plots, layout=layout)
fig.show()


In [78]:
import glob
points_list1 = glob.glob("../PointNet/shapenet-core-seg-less-subset1/Shapenetcore_benchmark/04379243/points/*.npy")
points_list2 = glob.glob("../PointNet/shapenet-core-seg-less-subset2/Shapenetcore_benchmark/04379243/points/*.npy")
print(len(points_list1))
print(len(points_list2))

120
120


In [79]:
import random
idx1 = random.randint(0,len(points_list1))

# load point cloud data
points = np.load(points_list1[idx1])
print(f"points shape = {points.shape}, min xyz = {np.min(points, axis=0)}, max xyz = {np.max(points, axis=0)}")

# load segmentation lables
seg_file_path = points_list1[idx1].replace('points', 'points_label').replace('.npy', '.seg')
seg_labels = np.loadtxt(seg_file_path).astype(np.int8)
print(f"seg_labels shape = {seg_labels.shape}, unique labels = {np.unique(seg_labels)}")

points shape = (2833, 3), min xyz = [-0.2303   0.      -0.39023], max xyz = [0.2303  0.      0.39015]
seg_labels shape = (2833,), unique labels = [1 2 3]


In [80]:
# there are max of 16 parts in an object in Shapenet core dataset
# creating random colors in according to part label
NUM_PARTS = 16
PART_COLORS = np.random.choice(range(255),size=(NUM_PARTS,3))

In [81]:
pc_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, color=PART_COLORS[seg_labels - 1], marker_size=2)
layout = dict(template="plotly_dark", title="Raw Point cloud", scene=PCD_SCENE, title_x=0.5)
fig = go.Figure(data=pc_plots, layout=layout)   
fig.show()

## Build a Custom Dataset
Now that we've explored our data, we'll get more 'PyTorch friendly' and create a Dataset object, as well as a PyTorch Dataloader.

In [82]:
class ShapeNetDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split_type, num_samples=2500):
        self.root_dir = root_dir
        self.split_type = split_type
        self.num_samples = num_samples
        with open(os.path.join(root_dir, f'{self.split_type}_split.json'), 'r') as f:
            self.split_data = json.load(f)       
            
    def __getitem__(self, index):
        # read point cloud data
        class_id, class_name, point_cloud_path, seg_label_path = self.split_data[index]
        
        # point cloud data
        point_cloud_path = os.path.join(self.root_dir, point_cloud_path)
        pc_data = np.load(point_cloud_path)
        
        # segmentation labels
        # -1 is to change part values from [1-16] to [0-15]
        # which helps when running segmentation
        pc_seg_labels = np.loadtxt(os.path.join(self.root_dir, seg_label_path)).astype(np.int8) - 1
#         pc_seg_labels = pc_seg_labels.reshape(pc_seg_labels.size,1)
        
        # Sample fixed number of points
        num_points = pc_data.shape[0]
        if num_points < self.num_samples:
            # Duplicate random points if the number of points is less than max_num_points
            additional_indices = np.random.choice(num_points, self.num_samples - num_points, replace=True)
            pc_data = np.concatenate((pc_data, pc_data[additional_indices]), axis=0)
            pc_seg_labels = np.concatenate((pc_seg_labels, pc_seg_labels[additional_indices]), axis=0)
                
        else:
            # Randomly sample max_num_points from the available points
            random_indices = np.random.choice(num_points, self.num_samples)
            pc_data = pc_data[random_indices]
            pc_seg_labels = pc_seg_labels[random_indices]
        
        # return variable
        data_dict= {}
        data_dict['class_id'] = class_id
        data_dict['class_name'] = class_name        
        data_dict['points'] = pc_data 
        data_dict['seg_labels'] = pc_seg_labels 
        return data_dict        
                    
    def __len__(self):
        return len(self.split_data)

In [83]:
train_set1 = ShapeNetDataset(root_dir = DATA_FOLDER1, split_type='train')
val_set1 = ShapeNetDataset(root_dir = DATA_FOLDER1, split_type='val')
test_set1 = ShapeNetDataset(root_dir = DATA_FOLDER1, split_type='test')
print(f"Train set 1 length = {len(train_set1)}")
print(f"Validation set 1 length = {len(val_set1)}")
print(f"Test set 1 length = {len(test_set1)}")

train_set2 = ShapeNetDataset(root_dir = DATA_FOLDER2, split_type='train')
val_set2 = ShapeNetDataset(root_dir = DATA_FOLDER2, split_type='val')
test_set2 = ShapeNetDataset(root_dir = DATA_FOLDER2, split_type='test')
print(f"Train set 2 length = {len(train_set1)}")
print(f"Validation 2 set length = {len(val_set1)}")
print(f"Test set 2 length = {len(test_set1)}")

Train set 1 length = 504
Validation set 1 length = 72
Test set 1 length = 144
Train set 2 length = 504
Validation 2 set length = 72
Test set 2 length = 144


In [84]:
data_dict= train_set2[25]
print(f"Keys in dataset sample = {list(data_dict.keys())}")
points = data_dict['points']
seg_labels = data_dict['seg_labels']
print(f"class_id = {data_dict['class_id']}, class_name = {data_dict['class_name']}")

Keys in dataset sample = ['class_id', 'class_name', 'points', 'seg_labels']
class_id = 0, class_name = Airplane


In [85]:
pc_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, color=PART_COLORS[seg_labels - 1], marker_size=2)
layout = dict(template="plotly_dark", title=f"{data_dict['class_name']}, class id = {data_dict['class_id']}, from Shapenetcore Torch Dataset", scene=PCD_SCENE, title_x=0.5)
fig = go.Figure(data=pc_plots, layout=layout)   
fig.show()

### Data loader for Custom dataset

In [86]:
def collate_fn(batch_list):
    ret = {}
    ret['class_id'] =  torch.from_numpy(np.array([x['class_id'] for x in batch_list])).long()
    ret['class_name'] = np.array([x['class_name'] for x in batch_list])
    ret['points'] = torch.from_numpy(np.stack([x['points'] for x in batch_list], axis=0)).float()
    ret['seg_labels'] = torch.from_numpy(np.stack([x['seg_labels'] for x in batch_list], axis=0)).long()
    return ret

In [87]:
sample_loader1 = torch.utils.data.DataLoader(train_set1, batch_size=16, num_workers=0, shuffle=True, collate_fn=collate_fn) 
dataloader_iter1 = iter(sample_loader1)   
batch_dict1 = next(dataloader_iter1)
print(batch_dict1.keys())
for key in ['points','seg_labels', 'class_id']:
    print(f"batch_dict[{key}].shape = {batch_dict1[key].shape}")

sample_loader2 = torch.utils.data.DataLoader(train_set2, batch_size=16, num_workers=0, shuffle=True, collate_fn=collate_fn) 
dataloader_iter2 = iter(sample_loader2)   
batch_dict2 = next(dataloader_iter2)
print(batch_dict2.keys())
for key in ['points','seg_labels', 'class_id']:
    print(f"batch_dict[{key}].shape = {batch_dict2[key].shape}")

dict_keys(['class_id', 'class_name', 'points', 'seg_labels'])
batch_dict[points].shape = torch.Size([16, 2500, 3])
batch_dict[seg_labels].shape = torch.Size([16, 2500])
batch_dict[class_id].shape = torch.Size([16])
dict_keys(['class_id', 'class_name', 'points', 'seg_labels'])
batch_dict[points].shape = torch.Size([16, 2500, 3])
batch_dict[seg_labels].shape = torch.Size([16, 2500])
batch_dict[class_id].shape = torch.Size([16])


In [88]:
batchSize= 64
workers = 0
train_loader1 = torch.utils.data.DataLoader(train_set1, batch_size=batchSize, shuffle=True, num_workers=workers, collate_fn=collate_fn)
val_loader1 = torch.utils.data.DataLoader(val_set1, batch_size=batchSize, shuffle=True, num_workers=workers, collate_fn=collate_fn)
test_loader1 = torch.utils.data.DataLoader(test_set1, batch_size=batchSize,shuffle=True, num_workers=workers, collate_fn=collate_fn)
train_loader2 = torch.utils.data.DataLoader(train_set2, batch_size=batchSize, shuffle=True, num_workers=workers, collate_fn=collate_fn)
val_loader2 = torch.utils.data.DataLoader(val_set2, batch_size=batchSize, shuffle=True, num_workers=workers, collate_fn=collate_fn)
test_loader2 = torch.utils.data.DataLoader(test_set2, batch_size=batchSize,shuffle=True, num_workers=workers, collate_fn=collate_fn)

# PointNet

**The model is PointNet.**<p>
![PointNet](https://d3i71xaburhd42.cloudfront.net/c3b6a072288e134f5ae6afe3eebc970ffb65cb68/3-Figure2-1.png)

It's coming from the paper: https://arxiv.org/pdf/1612.00593.pdf

The Point-Net will be done in several parts:
* T-Net
* FeatureNet
* Classification or Segmentation Head

### T-Net

In [89]:
import torch.nn.functional as F
from torch.autograd import Variable

class STN3d(nn.Module):
    """
    T-Net Model.
    STN stands for Spatial Transformer Network.
    """
    def __init__(self, num_points = 2500):
        super(STN3d, self).__init__()
        self.num_points = num_points
        self.conv1 = torch.nn.Conv1d(3, 64, 1)
        self.conv2 = torch.nn.Conv1d(64, 128, 1)
        self.conv3 = torch.nn.Conv1d(128, 1024, 1)
        
        self.mp1 = torch.nn.MaxPool1d(num_points)
        
        # FC layers
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 9)
        self.relu = nn.ReLU()

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)


    def forward(self, x):
        batchsize = x.size()[0]
        
        # Expected input shape = (bs, 3, num_points)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.mp1(x)
        x = x.view(-1, 1024)

        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)

        iden = Variable(torch.from_numpy(np.array([1,0,0,0,1,0,0,0,1]).astype(np.float32))).view(1,9).repeat(batchsize,1)
        if x.is_cuda:
            iden = iden.cuda()
        x = x + iden
        x = x.view(-1, 3, 3)
        return x

In [90]:
test_model = STN3d().to(device)
sim_data = Variable(torch.rand(32,3,2500)).to(device)
out = test_model(sim_data)
print('stn', out.size())

stn torch.Size([32, 3, 3])


## FeatureNet

In [91]:
class PointNetfeat(nn.Module):
    """
    This is the T-Net for Feature Transform.
    There is also MLP part 64,128,1024.
    """
    def __init__(self, num_points = 2500, global_feat = True):
        super(PointNetfeat, self).__init__()
        self.stn = STN3d(num_points = num_points)
        self.conv1 = torch.nn.Conv1d(3, 64, 1)
        self.conv2 = torch.nn.Conv1d(64, 128, 1)
        self.conv3 = torch.nn.Conv1d(128, 1024, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.mp1 = torch.nn.MaxPool1d(num_points)
        self.num_points = num_points
        self.global_feat = global_feat

    def forward(self, x):
        batchsize = x.size()[0]
        trans = self.stn(x)
        x = x.transpose(2,1)
        x = torch.bmm(x, trans)
        x = x.transpose(2,1)
        x = F.relu(self.bn1(self.conv1(x)))
        pointfeat = x
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = self.mp1(x)
        x = x.view(-1, 1024)
        if self.global_feat:
            return x, trans
        else:
            x = x.view(-1, 1024, 1).repeat(1, 1, self.num_points)
            return torch.cat([x, pointfeat], 1), trans

In [92]:
pointfeat = PointNetfeat(global_feat=True).to(device)
out, _ = pointfeat(sim_data)
print('global feat', out.size())

pointfeat = PointNetfeat(global_feat=False).to(device)
out, _ = pointfeat(sim_data)
print('point feat', out.size())

global feat torch.Size([32, 1024])
point feat torch.Size([32, 1088, 2500])


## Classifier Head

In [93]:
class PointNetCls(nn.Module):
    """
    Network for Classification: 512, 256, K.
    """
    def __init__(self, num_points = 2500, k = 2):
        super(PointNetCls, self).__init__()
        self.num_points = num_points
        self.feat = PointNetfeat(num_points, global_feat=True)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
    def forward(self, x):
        x, trans = self.feat(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return F.log_softmax(x, dim=-1), trans

In [94]:
cls = PointNetCls(k = 16).to(device)
out, _ = cls(sim_data)
print('class', out.size())

class torch.Size([32, 16])


## Training

In [95]:
def train_model(model, num_epochs, criterion, optimizer, dataloader_train,
                label_str = 'class_id', lr_scheduler = None, output_name = 'pointnet.pth'):
    # move model to device
    model.to(device)
    
    for epoch in range(num_epochs):
        print(f"Starting {epoch + 1} epoch ...")
        
        # Training
        model.train()
        train_loss = 0.0
        for batch_dict in tqdm(dataloader_train, total=len(dataloader_train)):            
            # Forward pass
            x = batch_dict['points'].transpose(1, 2).to(device)
            labels = batch_dict[label_str].to(device)
            pred, _ = model(x)
            loss = criterion(pred, labels)
            train_loss += loss.item()
              
            # Backward pass
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # adjust learning rate
            if lr_scheduler is not None:
                lr_scheduler.step()
            
        # compute per batch losses, metric value
        train_loss = train_loss / len(dataloader_train)

        print(f'Epoch: {epoch+1}, trainLoss:{train_loss:6.5f}')
    torch.save(model.state_dict(), output_name)

In [96]:
import torch.optim as optim

N_EPOCHS = 7
num_points = 2500
num_classes = 6
criterion = nn.NLLLoss()

# create model, optimizer, lr_scheduler and pass to training function
num_classes = len(class_id_name_map.items())

# Subset 1
classifier1 = PointNetCls(k=num_classes, num_points=num_points)
optimizer1 = optim.SGD(classifier1.parameters(), lr=0.01, momentum=0.9)
if torch.cuda.is_available():
    classifier1.cuda()

_ = train_model(classifier1, N_EPOCHS, criterion, optimizer1, train_loader1, output_name='pointnet1task1.pth')

# Subset 2
classifier2 = PointNetCls(k=num_classes, num_points=num_points)
optimizer2 = optim.SGD(classifier2.parameters(), lr=0.01, momentum=0.9)
if torch.cuda.is_available():
    classifier2.cuda()

_ = train_model(classifier2, N_EPOCHS, criterion, optimizer2, train_loader2, output_name='pointnet2task1.pth')


Starting 1 epoch ...


100%|██████████| 8/8 [00:49<00:00,  6.19s/it]


Epoch: 1, trainLoss:1.21727
Starting 2 epoch ...


100%|██████████| 8/8 [00:33<00:00,  4.17s/it]


Epoch: 2, trainLoss:0.59503
Starting 3 epoch ...


...

Epoch: 5, trainLoss:0.99076
Starting 6 epoch ...


100%|██████████| 8/8 [00:33<00:00,  4.13s/it]


Epoch: 6, trainLoss:0.89478
Starting 7 epoch ...


100%|██████████| 8/8 [00:34<00:00,  4.34s/it]

Epoch: 7, trainLoss:0.82154


## Inference

In [97]:
# --- Evaluate model trained on subset 1 ---
classifier1 = PointNetCls(k=num_classes).to(device)
classifier1.load_state_dict(torch.load('pointnet1task1.pth'))
classifier1.eval()

total_loss1 = 0.0
total_correct1 = 0
total_samples1 = 0

with torch.no_grad():
    for batch_dict in tqdm(test_loader1, total=len(test_loader1)):
        x = batch_dict['points'].transpose(1, 2).to(device)
        labels = batch_dict['class_id'].to(device)
        pred, _ = classifier1(x)

        # compute loss
        loss = criterion(pred, labels)
        total_loss1 += loss.item()

        # compute classification accuracy
        predicted_labels = pred.argmax(dim=1)  # (B,)
        total_correct1 += (predicted_labels == labels).sum().item()
        total_samples1 += labels.size(0)

evaluation_loss1 = total_loss1 / len(test_loader1)
accuracy1 = total_correct1 / total_samples1

print(f"Subset 1: Test loss = {evaluation_loss1:.4f}, Accuracy = {accuracy1*100:.2f}%")


# --- Evaluate model trained on subset 2 ---
classifier2 = PointNetCls(k=num_classes).to(device)
classifier2.load_state_dict(torch.load('pointnet2task1.pth'))
classifier2.eval()

total_loss2 = 0.0
total_correct2 = 0
total_samples2 = 0

with torch.no_grad():
    for batch_dict in tqdm(test_loader2, total=len(test_loader2)):
        x = batch_dict['points'].transpose(1, 2).to(device)
        labels = batch_dict['class_id'].to(device)
        pred, _ = classifier2(x)

        # compute loss
        loss = criterion(pred, labels)
        total_loss2 += loss.item()

        # compute classification accuracy
        predicted_labels = pred.argmax(dim=1)
        total_correct2 += (predicted_labels == labels).sum().item()
        total_samples2 += labels.size(0)

evaluation_loss2 = total_loss2 / len(test_loader2)
accuracy2 = total_correct2 / total_samples2

print(f"Subset 2: Test loss = {evaluation_loss2:.4f}, Accuracy = {accuracy2*100:.2f}%")


100%|██████████| 3/3 [00:04<00:00,  1.63s/it]


Subset 1: Test loss = 0.6366, Accuracy = 71.53%


100%|██████████| 3/3 [00:09<00:00,  3.15s/it]

Subset 2: Test loss = 2.2622, Accuracy = 33.33%


## Test on individual items

In [98]:
# Random test sample
test_sample = test_set1[np.random.choice(np.arange(len(test_set1)))]
batch_dict = collate_fn([test_sample])
x = batch_dict['points'].transpose(1, 2).to(device)

# Get model predictions
model_preds, _ = classifier1(x)
predicted_class = torch.argmax(model_preds, axis=1).detach().cpu().numpy()[0]
predicted_class_name = class_id_name_map[predicted_class]
pred_class_probs = F.softmax(model_preds.flatten(), dim=None).detach().cpu().numpy()

# plot results
title = f"Label = {test_sample['class_name']}, Predicted class = {predicted_class_name}"
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {}]], column_widths=[0.4, 0.6])
fig.update_layout(template="plotly_dark", scene=PCD_SCENE, height = 400, width = 1200,
                title=title, title_x=0.1, title_y=0.97, margin=dict(r=0, b=0, l=0, t=0))    
fig.add_trace(plot_pc_data3d(x=test_sample['points'][:,0], y=test_sample['points'][:,1], z=test_sample['points'][:,2]), row=1, col=1)
fig.add_trace(go.Bar(x=list(class_name_id_map.keys()), y=pred_class_probs, showlegend=False), row=1, col=2)
fig.show()

C:\Users\crnhr\AppData\Local\Temp\ipykernel_8784\2632083438.py:10: UserWarning:

Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.



In [99]:
# Random test sample
test_sample = test_set2[np.random.choice(np.arange(len(test_set2)))]
batch_dict = collate_fn([test_sample])
x = batch_dict['points'].transpose(1, 2).to(device)

# Get model predictions
model_preds, _ = classifier2(x)
predicted_class = torch.argmax(model_preds, axis=1).detach().cpu().numpy()[0]
predicted_class_name = class_id_name_map[predicted_class]
pred_class_probs = F.softmax(model_preds.flatten(), dim=None).detach().cpu().numpy()

# plot results
title = f"Label = {test_sample['class_name']}, Predicted class = {predicted_class_name}"
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {}]], column_widths=[0.4, 0.6])
fig.update_layout(template="plotly_dark", scene=PCD_SCENE, height = 400, width = 1200,
                title=title, title_x=0.1, title_y=0.97, margin=dict(r=0, b=0, l=0, t=0))    
fig.add_trace(plot_pc_data3d(x=test_sample['points'][:,0], y=test_sample['points'][:,1], z=test_sample['points'][:,2]), row=1, col=1)
fig.add_trace(go.Bar(x=list(class_name_id_map.keys()), y=pred_class_probs, showlegend=False), row=1, col=2)
fig.show()

C:\Users\crnhr\AppData\Local\Temp\ipykernel_8784\433066927.py:10: UserWarning:

Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.



## Segmentation

In [100]:
class PointNetDenseCls(nn.Module):
    """
    Network for Segmentation
    """
    def __init__(self, num_points = 2500, k = 2):
        super(PointNetDenseCls, self).__init__()
        self.num_points = num_points
        self.k = k
        self.feat = PointNetfeat(num_points, global_feat=False)
        self.conv1 = torch.nn.Conv1d(1088, 512, 1)
        self.conv2 = torch.nn.Conv1d(512, 256, 1)
        self.conv3 = torch.nn.Conv1d(256, 128, 1)
        self.conv4 = torch.nn.Conv1d(128, self.k, 1)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)

    def forward(self, x):
        batchsize = x.size()[0]
        x, trans = self.feat(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.conv4(x)
        return x, trans

In [101]:
seg = PointNetDenseCls(k = 16).to(device)
print(seg)
out, _ = seg(sim_data)
print('seg', out.size())

PointNetDenseCls(
  (feat): PointNetfeat(
    (stn): STN3d(
      (conv1): Conv1d(3, 64, kernel_size=(1,), stride=(1,))
      (conv2): Conv1d(64, 128, kernel_size=(1,), stride=(1,))
      (conv3): Conv1d(128, 1024, kernel_size=(1,), stride=(1,))
      (mp1): MaxPool1d(kernel_size=2500, stride=2500, padding=0, dilation=1, ceil_mode=False)
      (fc1): Linear(in_features=1024, out_features=512, bias=True)
      (fc2): Linear(in_features=512, out_features=256, bias=True)
      (fc3): Linear(in_features=256, out_features=9, bias=True)
      (relu): ReLU()
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn3): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True

## Training segmentation model

In [102]:
N_EPOCHS = 7
num_points = 2500
criterion = nn.CrossEntropyLoss()

# create model, optimizer, lr_scheduler and pass to training function
num_classes = len(class_id_name_map.items())
dense_classifier1 = PointNetDenseCls(k = NUM_PARTS, num_points = num_points)
dense_classifier1.to(device)

# DEFINE OPTIMIZERS
optimizer1 = optim.SGD(dense_classifier1.parameters(), lr=0.01, momentum=0.9)

train_model(dense_classifier1, N_EPOCHS, criterion, optimizer1, train_loader1, 
            label_str='seg_labels', output_name='pointnet_seg1task1.pth')

dense_classifier2 = PointNetDenseCls(k = NUM_PARTS, num_points = num_points)
dense_classifier2.to(device)

# DEFINE OPTIMIZERS
optimizer2 = optim.SGD(dense_classifier2.parameters(), lr=0.01, momentum=0.9)

train_model(dense_classifier2, N_EPOCHS, criterion, optimizer2, train_loader2, 
            label_str='seg_labels', output_name='pointnet_seg2task1.pth')

Starting 1 epoch ...


100%|██████████| 8/8 [02:20<00:00, 17.59s/it]


Epoch: 1, trainLoss:2.46757
Starting 2 epoch ...


100%|██████████| 8/8 [01:28<00:00, 11.00s/it]


Epoch: 2, trainLoss:1.28246
Starting 3 epoch ...


...

Epoch: 5, trainLoss:1.11031
Starting 6 epoch ...


100%|██████████| 8/8 [01:09<00:00,  8.69s/it]


Epoch: 6, trainLoss:1.08502
Starting 7 epoch ...


100%|██████████| 8/8 [01:10<00:00,  8.78s/it]

Epoch: 7, trainLoss:1.08258


## Inference

In [103]:
# --- Evaluate Dense Classifier for Subset 1 ---
dense_classifier1 = PointNetDenseCls(k=NUM_PARTS, num_points=num_points).to(device)
dense_classifier1.load_state_dict(torch.load('pointnet_seg1task1.pth'))
dense_classifier1.eval()

total_loss1 = 0.0
total_correct1 = 0
total_points1 = 0

with torch.no_grad():
    for batch_dict in tqdm(test_loader1, total=len(test_loader1)):
        x = batch_dict['points'].transpose(1, 2).to(device)
        labels = batch_dict['seg_labels'].to(device)
        pred, _ = dense_classifier1(x)  # pred: (B, k, N)

        # compute loss
        loss = criterion(pred, labels)
        total_loss1 += loss.item()

        # compute per-point accuracy
        predicted_labels = pred.argmax(dim=1)  # (B, N)
        total_correct1 += (predicted_labels == labels).sum().item()
        total_points1 += labels.numel()

evaluation_loss1 = total_loss1 / len(test_loader1)
accuracy1 = total_correct1 / total_points1
print(f"Subset 1: Test loss = {evaluation_loss1:.4f}, Accuracy = {accuracy1*100:.2f}%")

# --- Evaluate Dense Classifier for Subset 2 ---
dense_classifier2 = PointNetDenseCls(k=NUM_PARTS, num_points=num_points).to(device)
dense_classifier2.load_state_dict(torch.load('pointnet_seg2task1.pth'))
dense_classifier2.eval()

total_loss2 = 0.0
total_correct2 = 0
total_points2 = 0

with torch.no_grad():
    for batch_dict in tqdm(test_loader2, total=len(test_loader2)):
        x = batch_dict['points'].transpose(1, 2).to(device)
        labels = batch_dict['seg_labels'].to(device)
        pred, _ = dense_classifier2(x)

        # compute loss
        loss = criterion(pred, labels)
        total_loss2 += loss.item()

        # compute per-point accuracy
        predicted_labels = pred.argmax(dim=1)
        total_correct2 += (predicted_labels == labels).sum().item()
        total_points2 += labels.numel()

evaluation_loss2 = total_loss2 / len(test_loader2)
accuracy2 = total_correct2 / total_points2
print(f"Subset 2: Test loss = {evaluation_loss2:.4f}, Accuracy = {accuracy2*100:.2f}%")


100%|██████████| 3/3 [00:09<00:00,  3.16s/it]


Subset 1: Test loss = 0.9380, Accuracy = 62.94%


100%|██████████| 3/3 [00:08<00:00,  2.92s/it]

Subset 2: Test loss = 1.2080, Accuracy = 38.11%


## Test on individual items

In [104]:
# Random test sample
test_sample = test_set1[np.random.choice(np.arange(len(test_set1)))]
batch_dict = collate_fn([test_sample])

# Get model predictions
x = batch_dict['points'].transpose(1, 2).to(device)
model_preds, _ = dense_classifier1(x)
pred_part_labels = torch.argmax(model_preds, axis=1).detach().cpu().numpy()[0]

points = test_sample['points']
part_labels = test_sample['seg_labels']


# plot results
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]], column_widths=[0.5, 0.5],
                    subplot_titles=('Part Labels', 'Part Predictions'))

# ground truth part labels
gt_colors = [
    f'rgb({r},{g},{b})'
    for r, g, b in PART_COLORS[part_labels - 1]
]
part_label_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, 
                                  color=gt_colors, marker_size=2)

# ground truth part labels
pred_colors = [
    f'rgb({r},{g},{b})'
    for r, g, b in PART_COLORS[pred_part_labels - 1]
]

pred_part_label_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, 
                                  color=pred_colors, marker_size=2)

fig.update_layout(template="plotly_dark", scene=PCD_SCENE, scene2=PCD_SCENE, height = 400, width = 1200,
                title='PointNet Segmentation', title_x=0.5, title_y=0.97, margin=dict(r=0, b=0, l=0, t=0))
fig.add_trace(part_label_plots, row=1, col=1)
fig.add_trace(pred_part_label_plots, row=1, col=2)
fig.show()

In [105]:
# Random test sample
test_sample = test_set2[np.random.choice(np.arange(len(test_set2)))]
batch_dict = collate_fn([test_sample])

# Get model predictions
x = batch_dict['points'].transpose(1, 2).to(device)
model_preds, _ = dense_classifier2(x)
pred_part_labels = torch.argmax(model_preds, axis=1).detach().cpu().numpy()[0]

points = test_sample['points']
part_labels = test_sample['seg_labels']


# plot results
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]], column_widths=[0.5, 0.5],
                    subplot_titles=('Part Labels', 'Part Predictions'))

# ground truth part labels
gt_colors = [
    f'rgb({r},{g},{b})'
    for r, g, b in PART_COLORS[part_labels - 1]
]
part_label_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, 
                                  color=gt_colors, marker_size=2)

# ground truth part labels
pred_colors = [
    f'rgb({r},{g},{b})'
    for r, g, b in PART_COLORS[pred_part_labels - 1]
]

pred_part_label_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, 
                                  color=pred_colors, marker_size=2)

fig.update_layout(template="plotly_dark", scene=PCD_SCENE, scene2=PCD_SCENE, height = 400, width = 1200,
                title='PointNet Segmentation', title_x=0.5, title_y=0.97, margin=dict(r=0, b=0, l=0, t=0))
fig.add_trace(part_label_plots, row=1, col=1)
fig.add_trace(pred_part_label_plots, row=1, col=2)
fig.show()